# 19-Site RVB Dimer-Subspace Solver

This notebook runs the generalized eigenproblem in a maximum-dimer-covering basis:

`H c = E S c`, where `S_ij = <D_i|D_j>` and `H_ij = <D_i|H|D_j>`.

It saves optimized weighted-RVB coefficients and the optimized sector state, which can be used as the initial state for edge-colored HVA optimization.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

EXACT_ENERGY = float(np.load(PROJECT / "results" / "19site_fixed_sz_exact_n9.npz")["energy"])
plt.rcParams["figure.dpi"] = 130

## Run Subspace Solver

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT / "scripts" / "run_19site_rvb_subspace.py"),
    "--coverings",
    "54",
    "--checkpoints",
    "1,2,8,12,16,24,32,54",
]
result = subprocess.run(cmd, cwd=PROJECT, text=True, capture_output=True, check=True)
print(result.stdout)

## Energy and Fidelity Table

In [ ]:
df = pd.read_csv(PROJECT / "results" / "19site_rvb_subspace.csv")
df[[
    "coverings",
    "equal_energy",
    "equal_fidelity",
    "optimized_energy",
    "optimized_fidelity",
    "optimized_af_weight_participation",
    "optimized_max_magnetization",
]]

## Optimized RVB Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))

axes[0].plot(df["coverings"], df["equal_energy"], marker="o", label="equal-positive")
axes[0].plot(df["coverings"], df["optimized_energy"], marker="o", label="optimized")
axes[0].axhline(EXACT_ENERGY, color="black", ls="--", lw=1, label="exact")
axes[0].set_xlabel("Dimer coverings")
axes[0].set_ylabel("Energy")
axes[0].legend(fontsize=8)

axes[1].plot(df["coverings"], df["equal_fidelity"], marker="o", label="equal-positive")
axes[1].plot(df["coverings"], df["optimized_fidelity"], marker="o", label="optimized")
axes[1].set_xlabel("Dimer coverings")
axes[1].set_ylabel("Fidelity")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()